# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset is described by a Croissant-compliant schema, with multiple record sets, fields, and columns accessible programmatically.

### Dataset Source

- [FAIR² Croissant Schema (JSON-LD)](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure mlcroissant is available in this environment
!pip install mlcroissant

## 1. Data Loading

Let's load the metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's examine all available record sets and their fields. **All entities are referenced by their `@id`.**

We will print the list of record set `@id`s, and for each, the fields and columns with their `@id`.

In [ ]:
# List all record sets by @id
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    # Older versions: .record_set
    record_sets = getattr(metadata, 'record_set', [])
# Each record set is a Croissant RecordSet object
print(f"Found {len(record_sets)} record sets.")
for rs in record_sets:
    print(f"\nRecord set @id: {rs.id}")
    if hasattr(rs, 'fields'):
        print("  Fields / columns:")
        for fld in rs.fields:
            col = getattr(fld, 'column', None)
            col_id = col.id if col is not None and hasattr(col, 'id') else None
            print(f"    Field @id: {fld.id}", end='')
            if col_id:
                print(f"    (column @id: {col_id})")
            else:
                print("")


From the listed record sets above, pick the main data table for further analysis, referencing it by its `@id`. If the dataset only contains a single record set, we use that.

In [ ]:
# Extract record set IDs to use
record_set_ids = [rs.id for rs in record_sets]
print("Record set @id(s) available:", record_set_ids)
# Use the first record set as the main
main_record_set = record_set_ids[0]

## 3. Data Extraction

Now, let's load data from the selected record set(s) by their `@id` into Pandas DataFrames for analysis.

In [ ]:
# Load records from each record set into DataFrame, indexed by @id
dataframes = {}
for rs_id in record_set_ids:
    print(f"\nLoading records for record set @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Shape: {df.shape}")
# See available columns (should correspond to field/column @id)
print("\nColumns in main record set:")
print(dataframes[main_record_set].columns.tolist())
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field (by `@id`) for demonstration, filter the data, normalize the field, and group by a categorical field (by `@id`) if available.

> **Tip**: Always refer to fields/columns by their `@id` as shown in the overview.

In [ ]:
# Manually inspect the columns to select one numeric and one categorical field by @id
# For demonstration, try to infer typical field ids for medical/clinical tabular data

# Print columns again to select
print(dataframes[main_record_set].columns.tolist())

# Example field IDs (Replace with correct values if known, else fill with typicals)
numeric_field_id = None
categorical_field_id = None

# Attempt to infer a numeric field
for col in dataframes[main_record_set].columns:
    if any(s in col.lower() for s in ['age', 'interval', 'years', 'duration', 'tumor_size', 'months', 'n_size', 'number']):
        numeric_field_id = col
        break
if not numeric_field_id:
    # fallback: pick first numeric-looking column
    for col in dataframes[main_record_set].columns:
        if pd.api.types.is_numeric_dtype(dataframes[main_record_set][col]):
            numeric_field_id = col
            break
# Now, a categorical (group-by) field
for col in dataframes[main_record_set].columns:
    # Example keywords
    if any(s in col.lower() for s in ['sex', 'gender', 'msi', 'status', 'location', 'group', 'type', 'histology', 'metastasis']):
        categorical_field_id = col
        break
if not categorical_field_id:
    for col in dataframes[main_record_set].columns:
        if pd.api.types.is_object_dtype(dataframes[main_record_set][col]):
            categorical_field_id = col
            break
# Display selections
print(f"Numeric field @id selected: {numeric_field_id}")
print(f"Categorical field @id selected: {categorical_field_id}")

# Only proceed if both fields are present
if numeric_field_id and numeric_field_id in dataframes[main_record_set].columns:
    # Clean and convert numeric field if needed
    df = dataframes[main_record_set].copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()  # Use mean as a sample threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by categorical field
    if categorical_field_id and categorical_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(categorical_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {categorical_field_id}:")
        print(grouped_df.head())
else:
    print("Could not identify both numeric and categorical fields for EDA.")

## 5. Visualization

Let's visualize the selected numeric field's distribution, and compare its mean across groups (if a suitable categorical field is found).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric distribution
if numeric_field_id and numeric_field_id in dataframes[main_record_set].columns:
    plt.figure(figsize=(7,4))
    sns.histplot(dataframes[main_record_set][numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of numeric field (@id: {numeric_field_id})')
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if categorical_field_id and categorical_field_id in dataframes[main_record_set].columns:
        plt.figure(figsize=(7,5))
        sns.boxplot(x=dataframes[main_record_set][categorical_field_id], y=dataframes[main_record_set][numeric_field_id])
        plt.title(f'{numeric_field_id} by {categorical_field_id}')
        plt.xlabel(categorical_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- This notebook demonstrates how to load, explore, and analyze a Croissant-defined dataset with `mlcroissant`.
- We accessed all available entities using their `@id` and selected fields programmatically for flexible EDA.
- Further steps could include more advanced statistical modeling, cross-tabulation by clinical categories, or extending the data pipeline.

> **Note:** Ensure you have authorization or license to use the data and handle any personal or sensitive fields with care.